In [1]:
import sys, os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IS_KAGGLE:
    # Install packages not available on Kaggle
    %pip install -q kymatio kornia
    
    # Add repo to path (UPDATE 'deep-learning-course-project' to your dataset slug)
    repo_path = Path('/kaggle/input/deep-learning-course-project')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
else:
    # Local: add project root to path (assumes notebook is in notebooks/)
    project_root = Path.cwd().parent
    if (project_root / 'src').exists():
        sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import *
from src.models.architectures.RestNet18 import *
from src.models.architectures.ScatNet18 import *
from src.utils.training import *
from src.utils.visualization import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 2.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
====================== Hyperparameters =======================
N_EPOCHS: 200
T_MAX: 200
CRITERION: CrossEntropyLoss()
DEVICE: cuda
SEED: 42
BATCH_SIZE: 128
LR: 0.001
MOMENTUM: 0.9
WEIGHT_DECAY: 0.0001
Setting seed to 42


In [2]:
DEBUG = False
SKIP_TRAINING = False
EXP_NAME = "baseline_100"
print(f"Starting experiment {EXP_NAME}. DEBUG={DEBUG}, SKIP_TRAINING={SKIP_TRAINING}")

device = DEVICE

Starting experiment baseline_100. DEBUG=False, SKIP_TRAINING=False


In [3]:
resnet = MakeResNet18().to(device)
MODEL_NAME = "ResNet18"

total_params, model_size_mb = get_model_summary(resnet)
print(f"Total Parameters: {total_params:,}")
print(f"Model Size: {model_size_mb:.2f} MB")

Total Parameters: 11,173,962
Model Size: 42.63 MB


In [4]:
trainloader, valloader, testloader, train_set, val_set, test_set = get_cifar10_loaders_and_splits(
    n_samples_per_class_train=100
)
resnet_optimizer, resnet_scheduler = get_optimizer_and_scheduler(resnet)

100%|██████████| 170M/170M [00:02<00:00, 58.1MB/s]


Using default 500 samples per class for val.
Original train-val size: 50000
Train size: 1000
Val size: 5000
Test size: 10000
Samples per class (train): Counter({np.int64(7): 100, np.int64(1): 100, np.int64(8): 100, np.int64(4): 100, np.int64(3): 100, np.int64(9): 100, np.int64(5): 100, np.int64(0): 100, np.int64(2): 100, np.int64(6): 100})
Samples per class (val): Counter({np.int64(1): 500, np.int64(9): 500, np.int64(2): 500, np.int64(4): 500, np.int64(6): 500, np.int64(5): 500, np.int64(0): 500, np.int64(3): 500, np.int64(8): 500, np.int64(7): 500})
T_MAX: 200


In [5]:
if not SKIP_TRAINING:
    train_model(
        model=resnet,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=resnet_optimizer,
        scheduler=resnet_scheduler,
        device=device,
        experiment_name=EXP_NAME,
        model_name=MODEL_NAME,
        val_accuracy_storing_threshold=40,
        DEBUG=DEBUG
    )

Model weights will be saved to: /kaggle/working/artifacts/checkpoints/baseline_100_ResNet18.pth
Stats will be saved to: /kaggle/working/artifacts/stats/baseline_100_ResNet18.pkl

=== Starting Training: ResNet18 with 200 epochs ===
Epoch 1/200 | Loss: 4.305 | Val Acc: 10.00%
Epoch 2/200 | Loss: 3.575 | Val Acc: 13.22%
Epoch 3/200 | Loss: 2.414 | Val Acc: 11.76%
Epoch 4/200 | Loss: 2.259 | Val Acc: 13.72%
Epoch 5/200 | Loss: 2.260 | Val Acc: 20.20%
Epoch 6/200 | Loss: 2.104 | Val Acc: 22.92%
Epoch 7/200 | Loss: 2.039 | Val Acc: 21.36%
Epoch 8/200 | Loss: 1.993 | Val Acc: 26.58%
Epoch 9/200 | Loss: 1.977 | Val Acc: 21.26%
Epoch 10/200 | Loss: 1.930 | Val Acc: 26.00%
Epoch 11/200 | Loss: 1.842 | Val Acc: 28.00%
Epoch 12/200 | Loss: 1.834 | Val Acc: 28.56%
Epoch 13/200 | Loss: 1.771 | Val Acc: 31.32%
Epoch 14/200 | Loss: 1.745 | Val Acc: 29.68%
Epoch 15/200 | Loss: 1.719 | Val Acc: 31.54%
Epoch 16/200 | Loss: 1.730 | Val Acc: 31.70%
Epoch 17/200 | Loss: 1.676 | Val Acc: 34.00%
Epoch 18/200 

In [6]:
debug_suff = "_DEBUG" if DEBUG else ""
load_weights(resnet, experiment_name=EXP_NAME, model_name=(MODEL_NAME+ debug_suff), device=device)
if not SKIP_TRAINING:
    print(f'Final test accuracy is: {calculate_accuracy(resnet, testloader, device):.3f}')

Final test accuracy is: 48.060
